<a href="https://colab.research.google.com/github/hibbo67/ai-study-assistant/blob/main/algorithms_rag_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone Project: The Grokking Algorithms Assistant

## Problem
I have a PDF of "Grokking Algorithms" with hundreds of pages. I want to ask specific questions about the book (like "What is Big O?" or "How does quicksort work?") and get answers grounded strictly in the text, complete with the page number so I can read more.

## Users
Me, primarily. Later, anyone studying algorithms.

## Success looks like
- It answers questions accurately using the book's text.
- It refuses to answer questions not in the book (e.g., "What is the capital of France?").
- It tells me which page the answer came from.

## My 10 Test Questions (The Golden Set)
1. What is the run time for binary search? → Log time / O(log n)
2. How does selection sort work? → It takes O(n²) time, finds smallest and appends to new list.
3. What is the difference between an array and a linked list? → Arrays have fast reads, linked lists have fast inserts.
4. How does recursion work? → Base case and recursive case.
5. What is the worst-case run time of quicksort? → O(n²)
6. What is a hash table used for? → Lookups, caching, preventing duplicates.
7. What is a directed graph? → A graph where edges have arrows.
8. What is the Dijkstra's algorithm used for? → Finding the shortest path in a weighted graph.
9. What is a greedy algorithm? → At each step, pick the optimal move (locally optimal).
10. What is the knapsack problem? → Maximizing value given a weight constraint.

In [2]:
!pip install -q chromadb google-genai pypdf
from google.colab import userdata
from google import genai
from chromadb import Documents, EmbeddingFunction, Embeddings
import chromadb
import time

# 1. Setup Keys & Clients
api_keys = [userdata.get(f"GEMINI_KEY_{i}") for i in range(1, 8)]
ALL_KEYS = [k for k in api_keys if k]
embed_client = genai.Client(api_key=ALL_KEYS[0])

class GeminiEmbeddingFunction(EmbeddingFunction):
    def __init__(self): pass
    def __call__(self, input: Documents) -> Embeddings:
        vectors = []
        for text in input:
            result = embed_client.models.embed_content(
                model="gemini-embedding-2",
                contents=[text],
            )
            vectors.append([float(x) for x in result.embeddings[0].values])
        return vectors

# 2. Create Chroma Collection
db = chromadb.Client()
col = db.get_or_create_collection(name="algorithms_book", embedding_function=GeminiEmbeddingFunction())

# 3. The Text (I'm using a small chunk of the book to test)
# In a real app, you would use pypdf to extract this from a file.
book_text = """
Binary search is an algorithm; its input is a sorted list of elements. If an element you're looking for is in that list, binary search returns the position where it's located. Otherwise, binary search returns null.
Binary search will take log2 n steps to run in the worst case, whereas simple search will take n steps.
Big O notation is special notation that tells you how fast an algorithm is. Simple search needs to check each element, so it will take n operations. The run time in big O notation is O(n). Binary search needs log n operations to check a list of size n. Its running time is O(log n).
Selection sort is a neat algorithm, but it's not very fast. Quicksort is a faster sorting algorithm that only takes O(n log n) time.
A hash table maps keys to values. In the average case, hash tables take O(1) for everything. O(1) is called constant time.
Breadth-first search allows you to find the shortest distance between two things. It can answer: Is there a path from node A to node B? What is the shortest path from node A to node B?
If you have a negative-weight edge, you cannot use Dijkstra's algorithm. You must use the Bellman-Ford algorithm.
"""

# 4. Chunking (Paragraph based)
chunks = [p.strip() for p in book_text.split("\n") if len(p.strip()) > 20]

# 5. Add to Chroma
col.add(
    ids=[f"doc{i}" for i in range(len(chunks))],
    documents=chunks,
)

print(f"✅ Successfully ingested {col.count()} chunks into the database.")

✅ Successfully ingested 7 chunks into the database.


In [3]:
#The RAG Answer Function

def ask_book(question):
    # 1. Retrieve
    hits = col.query(query_texts=[question], n_results=2)
    retrieved_docs = hits["documents"][0]
    distances = hits["distances"][0]

    # 2. Guardrail
    if distances[0] > 0.75:
        return "I don't know. That information isn't in the book."

    # 3. Augment
    context = "\n\n".join(retrieved_docs)
    prompt = f"""Answer the user's question using ONLY the context below.
If the context does not contain the answer, you must say "I don't know."
Cite your sources using the bracketed numbers [1], [2], etc.

Context:
{context}

Question: {question}
"""

    # 4. Generate (with rotation)
    for key in ALL_KEYS:
        try:
            temp_client = genai.Client(api_key=key)
            response = temp_client.models.generate_content(
                model="gemini-3.6-flash",
                config={"thinking_config": {"thinking_level": "low"}},
                contents=prompt
            )
            return response.text
        except Exception as e:
            if "429" in str(e):
                time.sleep(1)
                continue
            raise e
    return "API limit reached."

# Test it!
print("Q: What is the run time for binary search?")
print(f"A: {ask_book('What is the run time for binary search?')}\n")

print("Q: What is the capital of France?")
print(f"A: {ask_book('What is the capital of France?')}\n")

Q: What is the run time for binary search?
A: The running time for binary search is O(log n) [1]. In the worst case, it will take $\log_2 n$ steps [1].

Q: What is the capital of France?
A: I don't know. That information isn't in the book.

